## Outline

- DF for minute features at date grain (Done)
- DF for daily features at date grain (Done)
- DF for returns over 1, 3 and 5 days (Done)
- Simple logistic, rfc and xgb models for daily alone, min alone and then combined
- Permutation importance
- Chart over rolling 5 days for 25 iterations, aka 6 months

In [31]:
import min_features, daily_return
import importlib
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.metrics import (
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.inspection import permutation_importance
import warnings
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

importlib.reload(min_features)
importlib.reload(daily_return)

df_min = min_features.min_features()
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 

df_main = pd.merge(df_min, df_daily, how='inner', on='Date')
df_main = df_main.sort_values(by='Date', ascending=False)

return_cols = df_main.columns[df_main.columns.str.contains("Return_")].to_list()
daily_cols = [
    c for c in df_daily.iloc[:, 1:].columns
    if "return" not in c.lower()
]
close_cols = df_min.columns[df_min.columns.str.contains("close_")].to_list()
min_cols = (
    df_min
    .loc[:, ~df_min.columns.isin(close_cols)]  # drop close_ columns
    .iloc[:, 1:]                               # drop first column
    .columns
    .to_list()
)
results_baseline = pd.read_csv("baseline_performance_1-3-5-10.csv")

In [ ]:
# -----------------------------
# Models
# -----------------------------
models = {
    "xgboost": XGBClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "random_forest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
}

# -----------------------------
# Helpers
# -----------------------------
def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else 0 #int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    #y_all = _to_binary(dfw[target_col].to_numpy())
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
            f"Run {k+1}/{runs} | "
            f"Train: {dates[train_start]} → {dates[train_end-1]} | "
            f"Test: {dates[test_start]} → {dates[test_end-1]} | "
            f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test  = X_all[test_start:test_end]
        y_test  = y_all[test_start:test_end]

        dist = _compute_dist(y_test)
        single_class_test = (np.unique(y_test).size < 2)

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)

            preds = m.predict(X_test)

            """
            # probabilities if available (for confidence metrics)
            proba = None
            if hasattr(m, "predict_proba"):
                proba = m.predict_proba(X_test)[:, 1]
            elif hasattr(m, "decision_function"):
                s = m.decision_function(X_test)
                # squash to (0,1) so confidence metrics work consistently
                proba = 1.0 / (1.0 + np.exp(-s))

            # confidence/coverage metrics (optional but useful)
            topk_acc = np.nan
            topk_cov = np.nan
            if proba is not None and len(proba) > 0:
                conf = np.abs(proba - 0.5)
                # top 40% by confidence (with 5 samples, this is ~2 samples)
                q = np.quantile(conf, 0.60)
                sel = conf >= q
                topk_cov = float(sel.mean())
                topk_acc = float((preds[sel] == y_test[sel]).mean()) if sel.any() else np.nan
            """
            rows.append({
                "run": k + 1,
                "model": model_name,
                "test_days": test_days,

                # core metrics
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "acc": float(accuracy_score(y_test, preds)),
                "sign_acc": 2 * float(accuracy_score(y_test, preds)) - 1,
                "mcc": float(matthews_corrcoef(y_test, preds)),

                # only meaningful if test has both classes
                "f1": np.nan if single_class_test else float(f1_score(y_test, preds, zero_division=0)),
                "precision": np.nan if single_class_test else float(precision_score(y_test, preds, zero_division=0)),
                "recall": np.nan if single_class_test else float(recall_score(y_test, preds, zero_division=0)),

                # confidence-conditioned performance (if proba/decision_function exists)
                #"top40_acc": topk_acc,
                #"top40_cov": topk_cov,

                **dist,

                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "train_years": train_years,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [2, 5, 10]#, 3, 5]  # add 3,5,etc later
train_years_grid = [4, 6]#[3, 5, 7]  # could be [3,4,5,6]
days_assessed = 228
test_days = [1, 2, 3, 4]
#runs = 75

results= []
results_df = pd.DataFrame()

for test_day in test_days:
    runs = days_assessed / test_day
    for feature_cols, feat_name in zip(column_sets, names):
        for r in returns:
            print(f"{r} | {feat_name}")
            target_col = f"Return_{r}"

            for train_years in train_years_grid:
                df_scores = walkback_runs(
                    df=df_main,
                    feature_cols=feature_cols,
                    target_col=target_col,
                    date_col="Date",
                    train_years=train_years,
                    test_days=test_day,
                    step_days=test_day,
                    runs=runs,
                    horizon_days=r,
                    purge_days=None,   # purge = horizon (safe default)
                    fill_inf=0.0,
                )

                df_scores["feature_set"] = feat_name
                df_scores["horizon"] = r

                results.append(df_scores)

results_df = pd.concat(results, ignore_index=True)

1 | daily
Run 1/75 | Train: 2023-01-03 → 2025-12-16 | Test: 2025-12-17 → 2025-12-19 | Train_n=735 | Test_n=3
Run 2/75 | Train: 2022-12-23 → 2025-12-09 | Test: 2025-12-10 → 2025-12-12 | Train_n=735 | Test_n=3
Run 3/75 | Train: 2022-12-16 → 2025-12-02 | Test: 2025-12-03 → 2025-12-05 | Train_n=735 | Test_n=3
Run 4/75 | Train: 2022-12-09 → 2025-11-21 | Test: 2025-11-24 → 2025-11-26 | Train_n=735 | Test_n=3
Run 5/75 | Train: 2022-12-02 → 2025-11-14 | Test: 2025-11-17 → 2025-11-19 | Train_n=735 | Test_n=3
Run 6/75 | Train: 2022-11-23 → 2025-11-07 | Test: 2025-11-10 → 2025-11-12 | Train_n=735 | Test_n=3
Run 7/75 | Train: 2022-11-16 → 2025-10-31 | Test: 2025-11-03 → 2025-11-05 | Train_n=735 | Test_n=3
Run 8/75 | Train: 2022-11-09 → 2025-10-24 | Test: 2025-10-27 → 2025-10-29 | Train_n=735 | Test_n=3
Run 9/75 | Train: 2022-11-02 → 2025-10-17 | Test: 2025-10-20 → 2025-10-22 | Train_n=735 | Test_n=3
Run 10/75 | Train: 2022-10-26 → 2025-10-10 | Test: 2025-10-13 → 2025-10-15 | Train_n=735 | Test_n=3

# TUNING

In [7]:
from sklearn.model_selection import ParameterSampler


# -----------------------------
# Models
# -----------------------------
models = {
    "xgboost_leaves": XGBClassifier(random_state=42),
    "xgboost_depth": XGBClassifier(random_state=42),
}

param_grids = {
    "xgboost_leaves": {
        'max_depth': [0],
        "max_leaves": [8, 16, 24, 32],
        'grow_policy': ['lossguide'],
        'learning_rate': [0.01, 0.05],    # Lower rates
        'min_child_weight': [6, 10],   # More conservative splits
        'subsample': [0.7, 0.8],     
        'colsample_bytree': [0.75, 0.85], 
        'colsample_bylevel': [0.7, 0.8],
        'colsample_bynode': [0.7, 0.8],            
        'n_estimators': [200, 400, 600],
    },
    "xgboost_depth": {
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.05],    # Lower rates
        'min_child_weight': [6, 10],   # More conservative splits
        'subsample': [0.7, 0.8, 0.9],     
        'colsample_bytree': [0.75, 0.85], 
        'colsample_bylevel': [0.7, 0.8],
        'colsample_bynode': [0.7, 0.8],    
        'gamma': [0.3, 0.5],         
        'alpha': [0.5, 1.0], 
        'lambda': [10, 15],            
        'n_estimators': [200, 400], 
    }
}

n_iter = 10
tune_seed = 42

def _compute_dist(y):
    """Distribution stats for y in {0,1}."""
    n = int(len(y))
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    return {
        "test_n": n,
        "test_pos_n": n_pos,
        "test_neg_n": n_neg,
        "test_pos_frac": (n_pos / n) if n else np.nan,
        "test_neg_frac": (n_neg / n) if n else np.nan,
    }

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,        # r (used for purge)
    purge_days=None,       # defaults to horizon_days
    fill_inf=0.0,
):
    """
    Deployment-aligned evaluation:
      - For each run, take a 5-day OOT test window stepping back by 5 days.
      - Train on the prior N years (fixed-length window) ending right before test.
      - Purge 'purge_days' from the end of train to avoid overlap leakage for forward-return labels.
      - Score ONLY on the OOT test window (distribution + metrics).
    Returns: long DataFrame with one row per (feature_set/run/model).
    """
    dfw = df.sort_values("Date").reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else 0 #int(horizon_days)

    X_all = dfw[safe_feature_cols].to_numpy()
    #y_all = _to_binary(dfw[target_col].to_numpy())
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy() if date_col in dfw.columns else None

    rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
            f"Run {k+1}/{runs} | "
            f"Train: {dates[train_start]} → {dates[train_end-1]} | "
            f"Test: {dates[test_start]} → {dates[test_end-1]} | "
            f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test  = X_all[test_start:test_end]
        y_test  = y_all[test_start:test_end]

        dist = _compute_dist(y_test)
        single_class_test = (np.unique(y_test).size < 2)

        for model_name, model in models.items():
            # inside your per-model loop, replace: m.fit(X_train, y_train)
            base = model
            grid = param_grids.get(model_name)

            if not grid:
                m = clone(base).fit(X_train, y_train)
            else:
                # simple time-respecting validation split from end of training
                val_n = max(50, int(0.2 * len(y_train)))
                X_tr, y_tr = X_train[:-val_n], y_train[:-val_n]
                X_va, y_va = X_train[-val_n:], y_train[-val_n:]

                best_params = None
                best_score = -np.inf

                for params in ParameterSampler(grid, n_iter=n_iter, random_state=tune_seed):
                    cand = clone(base).set_params(**params)
                    cand.fit(X_tr, y_tr)
                    pred_va = cand.predict(X_va)
                    score = balanced_accuracy_score(y_va, pred_va)

                    if score > best_score:
                        best_score = score
                        best_params = params

                # refit best on full training window (production-aligned)
                m = clone(base).set_params(**best_params).fit(X_train, y_train)

            preds = m.predict(X_test)

            # probabilities if available (for confidence metrics)
            proba = None
            if hasattr(m, "predict_proba"):
                proba = m.predict_proba(X_test)[:, 1]
            elif hasattr(m, "decision_function"):
                s = m.decision_function(X_test)
                # squash to (0,1) so confidence metrics work consistently
                proba = 1.0 / (1.0 + np.exp(-s))

            # confidence/coverage metrics (optional but useful)
            topk_acc = np.nan
            topk_cov = np.nan
            if proba is not None and len(proba) > 0:
                conf = np.abs(proba - 0.5)
                # top 40% by confidence (with 5 samples, this is ~2 samples)
                q = np.quantile(conf, 0.60)
                sel = conf >= q
                topk_cov = float(sel.mean())
                topk_acc = float((preds[sel] == y_test[sel]).mean()) if sel.any() else np.nan

            rows.append({
                "run": k + 1,
                "model": model_name,

                # core metrics
                "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                "acc": float(accuracy_score(y_test, preds)),
                "sign_acc": 2 * float(accuracy_score(y_test, preds)) - 1,
                "mcc": float(matthews_corrcoef(y_test, preds)),

                # only meaningful if test has both classes
                "f1": np.nan if single_class_test else float(f1_score(y_test, preds, zero_division=0)),
                "precision": np.nan if single_class_test else float(precision_score(y_test, preds, zero_division=0)),
                "recall": np.nan if single_class_test else float(recall_score(y_test, preds, zero_division=0)),

                # confidence-conditioned performance (if proba/decision_function exists)
                "top40_acc": topk_acc,
                "top40_cov": topk_cov,

                **dist,

                "train_n": int(len(y_train)),
                "train_start": dates[train_start] if dates is not None else train_start,
                "train_end": dates[train_end - 1] if dates is not None else train_end - 1,
                "test_start": dates[test_start] if dates is not None else test_start,
                "test_end": dates[test_end - 1] if dates is not None else test_end - 1,
                "train_years": train_years,
                "horizon_days": horizon_days,
                "n_features": len(safe_feature_cols),
            })

    return pd.DataFrame(rows)

# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols]#, daily_cols + min_cols]
names = ["daily", "minute"]#, "daily+minute"]

returns = [3, 5, 10]#, 3, 5]  # add 3,5,etc later
train_years_grid = [5]#[3, 5, 7]  # could be [3,4,5,6]
runs = 110
test_days = 5
step_days = 5

#results= []
#results_df = pd.DataFrame()

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        print(f"{r} | {feat_name}")
        target_col = f"Return_{r}"

        for train_years in train_years_grid:
            df_scores = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=None,   # purge = horizon (safe default)
                fill_inf=0.0,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            results.append(df_scores)

results_df = pd.concat(results, ignore_index=True)

3 | daily
Run 1/110 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
Run 2/110 | Train: 2021-01-08 → 2025-12-05 | Test: 2025-12-08 → 2025-12-12 | Train_n=1225 | Test_n=5
Run 3/110 | Train: 2020-12-31 → 2025-11-26 | Test: 2025-12-01 → 2025-12-05 | Train_n=1225 | Test_n=5
Run 4/110 | Train: 2020-12-22 → 2025-11-19 | Test: 2025-11-20 → 2025-11-26 | Train_n=1225 | Test_n=5
Run 5/110 | Train: 2020-12-15 → 2025-11-12 | Test: 2025-11-13 → 2025-11-19 | Train_n=1225 | Test_n=5
Run 6/110 | Train: 2020-12-08 → 2025-11-05 | Test: 2025-11-06 → 2025-11-12 | Train_n=1225 | Test_n=5
Run 7/110 | Train: 2020-12-01 → 2025-10-29 | Test: 2025-10-30 → 2025-11-05 | Train_n=1225 | Test_n=5
Run 8/110 | Train: 2020-11-20 → 2025-10-22 | Test: 2025-10-23 → 2025-10-29 | Train_n=1225 | Test_n=5
Run 9/110 | Train: 2020-11-13 → 2025-10-15 | Test: 2025-10-16 → 2025-10-22 | Train_n=1225 | Test_n=5
Run 10/110 | Train: 2020-11-06 → 2025-10-08 | Test: 2025-10-09 → 2025-10-15 | Tra

In [8]:
results_backup = results_df.copy()

In [19]:
results_df.to_csv("baseline_performance_1-3-5-10.csv", index=False)

In [ ]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
idx  = ["feature_set", "horizon", "train_years", "model"]

dfs = [results_df]
final_df = None
metric = 'acc' #'signed_acc, acc

for df in dfs:
    df = df.copy()

    # bucket to exact 5-day bins
    df["test_pos_frac"] = ((df["test_pos_frac"] * 5).round() / 5).clip(0, 1)

    # overall (distribution-agnostic)
    overall = (
        df.groupby(idx, as_index=False)
          .agg(m_all=(metric, "mean"), n_all=("run", "count"))
    )

    # by-bin
    g = (
        df.groupby(idx + ["test_pos_frac"], as_index=False)
          .agg(m=(metric, "mean"), n=("run", "count"))
    )

    wide = (
        g.pivot(index=idx, columns="test_pos_frac", values=["m", "n"])
         .reindex(columns=bins, level=1)
    )
    wide.columns = [f"{metric}_{frac:g}" for metric, frac in wide.columns]
    wide = wide.reset_index()

    # merge overall into wide
    wide = wide.merge(overall, on=idx, how="left")

    # optional: column order
    #column_order = ['feature_set', 'horizon', 'train_years', 'model', 'm_all', 'n_all', 'm_0', 'n_0', 'm_0.2', 
    #                'n_0.2', 'm_0.4', 'n_0.4', 'm_0.6', 'n_0.6', 'm_0.8', 'n_0.8', 'm_1', 'n_1']

    #wide = wide[column_order].round(3)

    if final_df is None:
        final_df = wide.copy()
    else:
        final_df = pd.concat([final_df, wide.copy()], ignore_index=True)

final_df[final_df['horizon'] == 10].sort_values(by=['horizon', 'm_all'], ascending=False)

,feature_set,horizon,train_years,model,m_0,m_0.4,m_0.6,m_1,n_0,n_0.4,n_0.6,n_1,m_all,n_all
30,daily,10,6,random_forest,0.629630,0.666667,0.555556,0.941667,18.0,11.0,6.0,40.0,0.795556,75
24,daily,10,3,random_forest,0.685185,0.606061,0.555556,0.891667,18.0,11.0,6.0,40.0,0.773333,75
28,daily,10,5,random_forest,0.648148,0.606061,0.555556,0.891667,18.0,11.0,6.0,40.0,0.764444,75
31,daily,10,6,xgboost,0.537037,0.606061,0.555556,0.916667,18.0,11.0,6.0,40.0,0.751111,75
61,daily+minute,10,5,xgboost,0.500000,0.636364,0.722222,0.891667,18.0,11.0,6.0,40.0,0.746667,75
26,daily,10,4,random_forest,0.648148,0.606061,0.500000,0.858333,18.0,11.0,6.0,40.0,0.742222,75
25,daily,10,3,xgboost,0.611111,0.545455,0.500000,0.866667,18.0,11.0,6.0,40.0,0.728889,75
27,daily,10,4,xgboost,0.462963,0.606061,0.555556,0.891667,18.0,11.0,6.0,40.0,0.720000,75
29,daily,10,5,xgboost,0.611111,0.545455,0.555556,0.841667,18.0,11.0,6.0,40.0,0.720000,75
58,daily+minute,10,4,random_forest,0.574074,0.515152,0.500000,0.875000,18.0,11.0,6.0,40.0,0.720000,75


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
)

def walkback_runs(
    df,
    feature_cols,
    target_col,
    *,
    models,                     # pass in models dict explicitly
    date_col="Date",
    train_years=6,
    test_days=5,
    step_days=5,
    runs=20,
    horizon_days=1,             # kept for metadata
    purge_days=None,            # default: 0 (prod-aligned)
    fill_inf=0.0,
    pi_tail_n=700,              # <-- PI computed on last N training records
    pi_scoring="balanced_accuracy",
    pi_repeats=5,
    pi_random_state=42,
    pi_n_jobs=-1,
):
    """
    Deployment-aligned evaluation:
      - For each run, take an OOT test window stepping back by step_days.
      - Train on prior N years ending right before test (optionally purged).
      - Score ONLY on OOT test.
      - Compute permutation importance on the *tail of training* (last pi_tail_n records).
    Returns:
      scores_df: one row per (run, model)
      pi_df: long PI table with one row per (run, model, feature)
    """
    dfw = df.sort_values(date_col).reset_index(drop=True).copy()

    # Drop any accidental return cols from features (belt+suspenders)
    safe_feature_cols = [c for c in feature_cols if "Return" not in c]

    # Basic numeric cleaning
    dfw[safe_feature_cols] = dfw[safe_feature_cols].replace([np.inf, -np.inf], fill_inf)

    n = len(dfw)
    train_size = 245 * int(train_years)
    test_size = int(test_days)
    step = int(step_days)
    purge = int(purge_days) if purge_days is not None else 0

    X_all = dfw[safe_feature_cols].to_numpy()
    y_all = dfw[target_col].to_numpy()
    dates = dfw[date_col].to_numpy()

    score_rows = []
    pi_rows = []

    for k in range(runs):
        test_end = n - k * step
        test_start = test_end - test_size
        if test_start < 0:
            break

        train_end = test_start - purge
        train_start = train_end - train_size
        if train_start < 0 or train_end <= train_start:
            break

        print(
            f"Run {k+1}/{runs} | "
            f"Train: {dates[train_start]} → {dates[train_end-1]} | "
            f"Test: {dates[test_start]} → {dates[test_end-1]} | "
            f"Train_n={train_end-train_start} | Test_n={test_end-test_start}"
        )

        X_train = X_all[train_start:train_end]
        y_train = y_all[train_start:train_end]
        X_test = X_all[test_start:test_end]
        y_test = y_all[test_start:test_end]

        single_class_test = (np.unique(y_test).size < 2)

        # PI window = last pi_tail_n of training
        tail_n = int(min(pi_tail_n, len(y_train)))
        X_pi = X_train[-tail_n:]
        y_pi = y_train[-tail_n:]

        for model_name, model in models.items():
            m = clone(model)
            m.fit(X_train, y_train)

            preds = m.predict(X_test)

            # optional: "top40" based on model output (not used for PI)
            proba = None
            if hasattr(m, "predict_proba"):
                proba = m.predict_proba(X_test)[:, 1]
            elif hasattr(m, "decision_function"):
                s = m.decision_function(X_test)
                proba = 1.0 / (1.0 + np.exp(-s))

            topk_acc = np.nan
            topk_cov = np.nan
            if proba is not None and len(proba) > 0:
                conf = np.abs(proba - 0.5)
                q = np.quantile(conf, 0.60)  # top 40%
                sel = conf >= q
                topk_cov = float(sel.mean())
                topk_acc = float((preds[sel] == y_test[sel]).mean()) if sel.any() else np.nan

            # --- permutation importance on TRAIN tail ---
            pi_mean = np.full(len(safe_feature_cols), np.nan, dtype=float)
            pi_std = np.full(len(safe_feature_cols), np.nan, dtype=float)

            # PI requires at least some label variability; if single-class, it can be meaningless
            if np.unique(y_pi).size >= 2 and tail_n >= 25:
                pi = permutation_importance(
                    m,
                    X_pi,
                    y_pi,
                    scoring=pi_scoring,
                    n_repeats=pi_repeats,
                    random_state=pi_random_state,
                    n_jobs=pi_n_jobs,
                )
                pi_mean = pi.importances_mean
                pi_std = pi.importances_std

                pi_rows.append(
                    pd.DataFrame(
                        {
                            "run": k + 1,
                            "model": model_name,
                            "feature": safe_feature_cols,
                            "pi_mean": pi_mean,
                            "pi_std": pi_std,
                            "pi_tail_n": tail_n,
                            "train_years": train_years,
                            "horizon_days": horizon_days,
                            "train_start": dates[train_start],
                            "train_end": dates[train_end - 1],
                            "test_start": dates[test_start],
                            "test_end": dates[test_end - 1],
                        }
                    )
                )

            score_rows.append(
                {
                    "run": k + 1,
                    "model": model_name,
                    "bal_acc": float(balanced_accuracy_score(y_test, preds)),
                    "acc": float(accuracy_score(y_test, preds)),
                    "sign_acc": 2.0 * float(accuracy_score(y_test, preds)) - 1.0,
                    "mcc": float(matthews_corrcoef(y_test, preds)),
                    "f1": np.nan if single_class_test else float(f1_score(y_test, preds, zero_division=0)),
                    "precision": np.nan if single_class_test else float(precision_score(y_test, preds, zero_division=0)),
                    "recall": np.nan if single_class_test else float(recall_score(y_test, preds, zero_division=0)),
                    "top40_acc": topk_acc,
                    "top40_cov": topk_cov,
                    "train_n": int(len(y_train)),
                    "test_n": int(len(y_test)),
                    "train_start": dates[train_start],
                    "train_end": dates[train_end - 1],
                    "test_start": dates[test_start],
                    "test_end": dates[test_end - 1],
                    "train_years": train_years,
                    "horizon_days": horizon_days,
                    "n_features": len(safe_feature_cols),
                    "pi_tail_n": tail_n,
                }
            )

    scores_df = pd.DataFrame(score_rows)
    pi_df = pd.concat(pi_rows, ignore_index=True) if len(pi_rows) else pd.DataFrame(
        columns=["run", "model", "feature", "pi_mean", "pi_std", "pi_tail_n",
                 "train_years", "horizon_days", "train_start", "train_end", "test_start", "test_end"]
    )
    return scores_df, pi_df


# -----------------------------
# Run grid (feature sets x horizon x train_years, etc.)
# -----------------------------
column_sets = [daily_cols, min_cols, daily_cols + min_cols]
names = ["daily", "minute", "daily+minute"]

returns = [1, 3, 5, 10]
train_years_grid = [5]
runs = 5
test_days = 5
step_days = 50

scores_all = []
pi_all = []

for feature_cols, feat_name in zip(column_sets, names):
    for r in returns:
        target_col = f"Return_{r}"
        print(f"{r} | {feat_name}")

        for train_years in train_years_grid:
            df_scores, df_pi = walkback_runs(
                df=df_main,
                feature_cols=feature_cols,
                target_col=target_col,
                models=models,
                date_col="Date",
                train_years=train_years,
                test_days=test_days,
                step_days=step_days,
                runs=runs,
                horizon_days=r,
                purge_days=None,   # prod-aligned
                fill_inf=0.0,
                pi_tail_n=700,
                pi_scoring="balanced_accuracy",
                pi_repeats=10,
                pi_random_state=42,
                pi_n_jobs=-1,
            )

            df_scores["feature_set"] = feat_name
            df_scores["horizon"] = r

            df_pi["feature_set"] = feat_name
            df_pi["horizon"] = r

            scores_all.append(df_scores)
            pi_all.append(df_pi)

results_pi_df = pd.concat(scores_all, ignore_index=True)
pi_long_df = pd.concat(pi_all, ignore_index=True)


1 | daily
Run 1/5 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
Run 2/5 | Train: 2020-10-30 → 2025-10-01 | Test: 2025-10-02 → 2025-10-08 | Train_n=1225 | Test_n=5
Run 3/5 | Train: 2020-08-20 → 2025-07-22 | Test: 2025-07-23 → 2025-07-29 | Train_n=1225 | Test_n=5
Run 4/5 | Train: 2020-06-10 → 2025-05-07 | Test: 2025-05-08 → 2025-05-14 | Train_n=1225 | Test_n=5
Run 5/5 | Train: 2020-03-30 → 2025-02-25 | Test: 2025-02-26 → 2025-03-04 | Train_n=1225 | Test_n=5
3 | daily
Run 1/5 | Train: 2021-01-15 → 2025-12-12 | Test: 2025-12-15 → 2025-12-19 | Train_n=1225 | Test_n=5
Run 2/5 | Train: 2020-10-30 → 2025-10-01 | Test: 2025-10-02 → 2025-10-08 | Train_n=1225 | Test_n=5
Run 3/5 | Train: 2020-08-20 → 2025-07-22 | Test: 2025-07-23 → 2025-07-29 | Train_n=1225 | Test_n=5
Run 4/5 | Train: 2020-06-10 → 2025-05-07 | Test: 2025-05-08 → 2025-05-14 | Train_n=1225 | Test_n=5
Run 5/5 | Train: 2020-03-30 → 2025-02-25 | Test: 2025-02-26 → 2025-03-04 | Train_n=1225 |